# 05.02 基于图邻接表向 CSR 转换的并行 SSSP 算子开发

本 Notebook 是本章的**核心学习与实验入口**。将从一个空的工作目录开始，在 code cell 中依次创建测试数据工具、Tiling 数据结构、Ascend C Kernel、Host 调度程序、构建脚本和精度验证程序，最后完成编译、执行。

目标环境固定为 **CANN 9.0.0、Atlas 910B3/A2、ARM64**。

## 学习目标与完整路线

完成实验后，你应当能够：

1. 解释为何普通出边 CSR 要转换为按目标顶点组织的转置 CSR。
2. 写出同步 Pull Bellman-Ford 的单轮更新，并说明双缓冲的必要性。
3. 根据顶点数、AI Core 数和 UB 容量设计 Tiling 参数。
4. 理解 Host 与 Kernel 的职责。
5. 使用 CMake 构建程序，并用 CPU Dijkstra Golden 验证 FP32 精度。

实验路线为：**创建工作区 → 构造数据 → 设计 Tiling → 编写 Kernel → 编写 Host → 配置构建 → 编译运行 → 精度与异常验收**。请按顺序执行全部 code cell，因为后面的文件路径依赖前面的工作目录初始化。

## 0. 创建 Notebook 工作区

`05.02_parallel_sssp.ipynb` 的 `%%writefile` 单元保存完整工程内容；`work/05.02_parallel_sssp` 是执行这些单元后生成的实验工程。

可以修改、破坏甚至删除 `work`，然后重新运行 Notebook 恢复完整工程。所有后续 `%%writefile` 都写入当前 Jupyter 工作目录下的固定 `work` 子目录。

### 本步骤关键点

- 使用当前 Jupyter 工作目录，不再额外判断仓库或章节路径。
- 重复运行时，后续 `%%writefile` 会明确覆盖对应文件。
- `build` 同时保存数据文件与 CMake 构建产物。

In [ ]:
from pathlib import Path

import os
import shutil
import subprocess
import sys

# Ascend C 的自动生成 CMake 文件会解析源码绝对路径。
# 为避免中文路径被转义成非法 CMake 字符串，实际编译工程放到 data_structures_compute 下的纯英文目录。
cwd = Path.cwd().resolve()
chapter_name = "05_图邻接表向CSR稀疏张量的格式转换"
if (cwd / "05.02_parallel_sssp.ipynb").exists():
    CHAPTER_DIR = cwd
elif (cwd / chapter_name / "05.02_parallel_sssp.ipynb").exists():
    CHAPTER_DIR = cwd / chapter_name
else:
    raise FileNotFoundError("请从 data_structures_compute 目录或 05 章节目录启动 Notebook")
COURSE_DIR = CHAPTER_DIR.parent

if os.environ.get("SSSP_WORK_DIR"):
    WORK_DIR = Path(os.environ["SSSP_WORK_DIR"]).expanduser().resolve()
else:
    WORK_DIR = (COURSE_DIR / "work" / "05.02_parallel_sssp").resolve()
WRITE_ROOT = WORK_DIR.as_posix()

print("[步骤 0] 创建实验工作区")
print("章节目录：", CHAPTER_DIR)
print("课程目录：", COURSE_DIR)
print("工作区：", WORK_DIR)
print("写入根路径：", WRITE_ROOT)

for relative in ("op_kernel", "op_host", "scripts", "build"):
    (WORK_DIR / relative).mkdir(parents=True, exist_ok=True)
for legacy in (
    WORK_DIR / "op_kernel" / "parallel_sssp_kernel.cpp",
    WORK_DIR / "op_host" / "parallel_sssp_main.cpp",
):
    if legacy.exists():
        legacy.unlink()

print("源码目录：", WORK_DIR / "op_kernel", "和", WORK_DIR / "op_host")
print("构建目录：", WORK_DIR / "build")
print("状态：目录准备完成，后续 %%writefile 单元将在这个纯英文路径生成工程文件。")

## 1. 检查 CANNLab 开发环境

Ascend C 开发依赖 CANN Toolkit、算子包 OPP、编译器以及 NPU。`ASCEND_HOME_PATH` 是 Toolkit 根目录，`ASCEND_OPP_PATH` 指向算子实现包，`npu-smi info` 用于确认设备型号和健康状态。

在目标 CANNLab 中应看到两个路径和至少一张 910B3 设备。本地没有 NPU 时，本单元只给出提示，不会中断后续 CPU 部分。

In [ ]:
ASCEND_HOME = os.environ.get("ASCEND_HOME_PATH")
ASCEND_OPP = os.environ.get("ASCEND_OPP_PATH")
NPU_SMI = shutil.which("npu-smi")

print("[步骤 1] 检查 CANN Toolkit 与 NPU")
print("ASCEND_HOME_PATH =", ASCEND_HOME or "<not set>")
print("ASCEND_OPP_PATH  =", ASCEND_OPP or "<not set>")
print("npu-smi          =", NPU_SMI or "<not found>")
if NPU_SMI:
    print("调用：npu-smi info")
    subprocess.run([NPU_SMI, "info"], check=False)
    print("状态：检测到 NPU，后续可以编译并运行算子。")
else:
    print("当前环境没有 NPU；先完成 CPU 数据与静态部分，NPU 单元会自动跳过。")

## 2. 明确算子接口与工程边界

单轮 Kernel 接收以下张量：

<table
  align="left"
  style="width: 60%;
         max-width: 1200px;
         margin: 0 auto 0 0 !important;
         text-align: left;">
  <thead>
    <tr>
      <th style="text-align: left;">参数</th>
      <th style="text-align: left;">类型与形状</th>
      <th style="text-align: left;">读写</th>
      <th style="text-align: left;">作用</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="text-align: left;"><code>row_ptr</code></td>
      <td style="text-align: left;"><code>int32[V+1]</code></td>
      <td style="text-align: left;">只读</td>
      <td style="text-align: left;">转置 CSR 中每个目标顶点的入边区间</td>
    </tr>
    <tr>
      <td style="text-align: left;"><code>src_idx</code></td>
      <td style="text-align: left;"><code>int32[E]</code></td>
      <td style="text-align: left;">只读</td>
      <td style="text-align: left;">每条入边的源顶点编号</td>
    </tr>
    <tr>
      <td style="text-align: left;"><code>weights</code></td>
      <td style="text-align: left;"><code>float32[E]</code></td>
      <td style="text-align: left;">只读</td>
      <td style="text-align: left;">非负有限边权</td>
    </tr>
    <tr>
      <td style="text-align: left;"><code>dist_in</code></td>
      <td style="text-align: left;"><code>float32[V]</code></td>
      <td style="text-align: left;">只读</td>
      <td style="text-align: left;">上一轮距离快照</td>
    </tr>
    <tr>
      <td style="text-align: left;"><code>dist_out</code></td>
      <td style="text-align: left;"><code>float32[V]</code></td>
      <td style="text-align: left;">只写</td>
      <td style="text-align: left;">本轮距离结果</td>
    </tr>
    <tr>
      <td style="text-align: left;"><code>tiling</code></td>
      <td style="text-align: left;">结构体</td>
      <td style="text-align: left;">只读</td>
      <td style="text-align: left;">顶点数、边数和分核参数</td>
    </tr>
  </tbody>
</table>
<div style="clear: both;"></div>

Host 将源点初始化为 `0`，其余顶点初始化为 `1e30`，最多启动 `V-1` 轮 Kernel。每轮结束后同步 stream 并交换 `dist_in/dist_out`，从而保持严格的同步 Bellman-Ford 语义。

## 3. 编写图数据与 CPU Golden 工具

`graph_utils.py` 是实验的数据处理中心，它负责四件事：校验图输入、将图邻接表整理为出边 CSR、构造适合 Pull 的转置 CSR、使用 CPU Dijkstra 计算 Golden。

### 为什么先从邻接表开始

数据结构课程中最常见的图表示是邻接表：`adj[u]=[(v,w), ...]` 表示从源顶点 `u` 出发可以到达若干目标顶点 `v`，边权为 `w`。邻接表直观、便于增删边，但每个顶点的边数不同，内存中不是统一长度的连续块；NPU Kernel 更适合读取连续数组。因此本实验先把邻接表转换为 CSR，再把出边 CSR 转换为转置 CSR。

### 为什么从邻接表生成两种 CSR

- **普通出边 CSR** 仍按源顶点组织邻接边，适合 CPU Dijkstra 从当前顶点扩展后继，用来生成可信参考答案。
- **转置 CSR** 按目标顶点组织入边，把目标顶点 `v` 的所有前驱 `u` 放在同一行，NPU Pull Kernel 可以让一个 AI Core 独占 `v` 的写回权。

边权必须是非负有限 `float32`；顶点编号必须在 `[0,V)`；同一目标顶点的入边连续存放。下面先用一个小图演示邻接表到 CSR/转置 CSR 的格式变化，再用 `%%writefile` 创建正式实验工具。

In [ ]:
%%writefile $WRITE_ROOT/scripts/graph_utils.py
from __future__ import annotations

import heapq
import math
from typing import Iterable

import numpy as np

INF = np.float32(1.0e30)
MAX_VERTEX_NUM = 16384


# 在构造任意图存储格式前统一检查顶点范围与边权约束。
def validate_edges(vertex_count: int, edges: Iterable[tuple[int, int, float]]):
    if not 1 <= vertex_count <= MAX_VERTEX_NUM:
        raise ValueError("vertex_count is outside the supported range")
    result = []
    for u, v, weight in edges:
        if not (0 <= u < vertex_count and 0 <= v < vertex_count):
            raise ValueError("edge contains an invalid vertex")
        if not math.isfinite(weight) or weight < 0:
            raise ValueError("weights must be finite and nonnegative")
        result.append((int(u), int(v), float(weight)))
    return result


# 边列表先整理为邻接表，这是数据结构课程中最直观的图表示。
def edges_to_adjacency(vertex_count: int, edges):
    adjacency = [[] for _ in range(vertex_count)]
    for u, v, weight in validate_edges(vertex_count, edges):
        adjacency[u].append((v, weight))
    return adjacency


# 邻接表 -> 出边 CSR：按源顶点 u 连续存放所有后继 v。
def adjacency_to_outgoing_csr(adjacency):
    row_ptr, col_idx, weights = [0], [], []
    for row in adjacency:
        row = sorted(row, key=lambda item: item[0])
        col_idx.extend(v for v, _ in row)
        weights.extend(w for _, w in row)
        row_ptr.append(len(col_idx))
    return (np.asarray(row_ptr, np.int32), np.asarray(col_idx, np.int32),
            np.asarray(weights, np.float32))


# 邻接表 -> 转置 CSR：按目标顶点 v 连续存放所有前驱 u，供 Pull Kernel 使用。
def adjacency_to_transposed_csr(adjacency):
    vertex_count = len(adjacency)
    incoming = [[] for _ in range(vertex_count)]
    for u, row in enumerate(adjacency):
        for v, weight in row:
            incoming[v].append((u, weight))
    row_ptr, src_idx, weights = [0], [], []
    for row in incoming:
        row.sort(key=lambda item: item[0])
        src_idx.extend(u for u, _ in row)
        weights.extend(w for _, w in row)
        row_ptr.append(len(src_idx))
    return (np.asarray(row_ptr, np.int32), np.asarray(src_idx, np.int32),
            np.asarray(weights, np.float32))


# 出边 CSR 按源顶点组织邻接边，便于理解原始图结构并供 CPU 侧算法使用。
def outgoing_csr(vertex_count: int, edges):
    adjacency = edges_to_adjacency(vertex_count, edges)
    return adjacency_to_outgoing_csr(adjacency)


# 转置 CSR 按目标顶点组织入边，正好对应 NPU Pull Kernel 的逐顶点读取方式。
def transpose_csr(vertex_count: int, edges):
    adjacency = edges_to_adjacency(vertex_count, edges)
    return adjacency_to_transposed_csr(adjacency)


# Dijkstra 独立计算可信参考结果，用于判断 NPU 输出是否正确。
def dijkstra_golden(vertex_count: int, edges, source: int):
    if not 0 <= source < vertex_count:
        raise ValueError("source is outside the graph")
    adjacency = edges_to_adjacency(vertex_count, edges)
    distance = [float(INF)] * vertex_count
    distance[source] = 0.0
    queue = [(0.0, source)]
    while queue:
        current, u = heapq.heappop(queue)
        # 同一顶点可能多次入堆，跳过已经失效的较长路径记录。
        if current != distance[u]:
            continue
        for v, weight in adjacency[u]:
            candidate = current + weight
            if candidate < distance[v]:
                distance[v] = candidate
                heapq.heappush(queue, (candidate, v))
    return np.asarray(distance, dtype=np.float32)


### 编写测试数据生成器

`gen_data.py` 根据案例名称生成确定的测试图，并输出 row_ptr.bin（记录各目标顶点入边的起止位置）、src_idx.bin（记录每条入边的源顶点）、weights.bin（记录对应边权）、golden.bin（保存 CPU Dijkstra 计算的标准最短距离，用于验证 NPU 输出）和 meta.txt（记录顶点数 V、边数 E 与源点 source）。

随机图采用固定随机种子，保证每次生成相同数据，便于复现和排查问题。

二进制文件直接采用 `int32/float32` 连续布局。测试集覆盖单顶点、链、不可达顶点、零权、多条等长最短路、稀疏随机图和较密图。

In [ ]:
%%writefile $WRITE_ROOT/scripts/gen_data.py
from __future__ import annotations

import argparse
from pathlib import Path

import numpy as np

from graph_utils import dijkstra_golden, transpose_csr


def make_case(name: str):
    # 每个案例分别覆盖一种边界条件或图结构特征。
    if name == "single":
        return 1, [], 0
    if name == "chain":
        return 16, [(i, i + 1, float(i % 3 + 1)) for i in range(15)], 0
    if name == "disconnected":
        return 7, [(0, 1, 2), (1, 2, 3), (4, 5, 1)], 0
    if name == "zero_weight":
        return 5, [(0, 1, 0), (1, 2, 0), (0, 2, 2), (2, 3, 1)], 0
    if name == "tie":
        return 5, [(0, 1, 1), (0, 2, 1), (1, 3, 2), (2, 3, 2), (3, 4, 1)], 0
    if name in {"random_sparse", "random_dense"}:
        rng = np.random.default_rng(20260718 if name == "random_sparse" else 20260719)
        vertex_count = 64
        probability = 0.06 if name == "random_sparse" else 0.28
        edges = []
        for u in range(vertex_count):
            for v in range(vertex_count):
                if u != v and rng.random() < probability:
                    edges.append((u, v, float(rng.uniform(0.0, 10.0))))
        return vertex_count, edges, 0
    return 6, [(0, 1, 2), (0, 2, 5), (1, 2, 1), (1, 3, 4),
               (2, 3, 1), (3, 4, 3), (1, 4, 10)], 0


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--case", default="demo")
    args = parser.parse_args()
    vertex_count, edges, source = make_case(args.case)
    # NPU 的 Pull 计算使用入边（转置）CSR，CPU Dijkstra 负责生成 Golden 参考结果。
    row_ptr, src_idx, weights = transpose_csr(vertex_count, edges)
    golden = dijkstra_golden(vertex_count, edges, source)
    Path("input").mkdir(exist_ok=True)
    Path("output").mkdir(exist_ok=True)
    # 二进制文件的数据类型和排列方式与 C++ Host 的 int32/float32 读取接口保持一致。
    row_ptr.tofile("input/row_ptr.bin")
    src_idx.tofile("input/src_idx.bin")
    weights.tofile("input/weights.bin")
    golden.tofile("output/golden.bin")
    Path("meta.txt").write_text(
        f"{vertex_count} {len(edges)} {source}\n", encoding="utf-8")
    print(f"case={args.case}, V={vertex_count}, E={len(edges)}, source={source}")
    print("incoming row_ptr:", row_ptr.tolist())
    print("golden:", golden.tolist())


if __name__ == "__main__":
    main()

### 先在 CPU 上验证数据链路

生成 `demo` 数据，并用 NumPy 读回转置 CSR 与 Golden。先确认文件长度、类型和 CSR 区间正确，可以避免把数据问题误判为 Kernel 问题。

In [ ]:
# 数据生成脚本在 build 目录运行，因此输入、Golden 和元数据会写入同一运行目录。
demo_dir = WORK_DIR / "build"
generator = WORK_DIR / "scripts" / "gen_data.py"
print("[步骤 3.1] 调用测试数据生成器")
print("脚本：", generator)
print("案例：demo")
subprocess.run(
    [sys.executable, str(generator), "--case", "demo"],
    cwd=demo_dir,
    check=True,
)

# 按 Host 约定的数据类型读取二进制文件，检查磁盘布局是否与算子接口一致。
import numpy as np
row_ptr = np.fromfile(demo_dir / "input" / "row_ptr.bin", dtype=np.int32)
src_idx = np.fromfile(demo_dir / "input" / "src_idx.bin", dtype=np.int32)
weights = np.fromfile(demo_dir / "input" / "weights.bin", dtype=np.float32)
golden = np.fromfile(demo_dir / "output" / "golden.bin", dtype=np.float32)
print("\n[步骤 3.2] 读取生成的二进制文件")
for relative in (
    "input/row_ptr.bin", "input/src_idx.bin", "input/weights.bin",
    "output/golden.bin", "meta.txt",
):
    path = demo_dir / relative
    print(f"{relative:24s} {path.stat().st_size:6d} bytes")
print("\n[步骤 3.3] 展示转置 CSR 与 CPU Golden")
print("row_ptr =", row_ptr)
print("src_idx =", src_idx)
print("weights =", weights)
print("golden  =", golden)
assert row_ptr[-1] == len(src_idx) == len(weights)

## 4. 设计 Tiling 与 UB 预算

### Tiling 是什么

Tiling 是 Host 在 launch 前计算、Kernel 启动后读取的小型控制数据。这里按**目标顶点**分核：`blockNum=min(V, 可用 Vector Core 数)`，`verticesPerCore=ceil(V/blockNum)`，尾核再用 `min` 截断。

### 为什么按目标顶点分核

每个目标顶点只属于一个 AI Core，因而 `dist_out[v]` 只有一个写者，但是高入度顶点可能造成负载不均。

### UB 规划

- 完整 `dist_in` 快照：`MAX_VERTEX_NUM * 4 = 64 KiB`。
- 257 个行偏移按 32B 对齐到 264 个元素：1056 B。
- 一个边块的索引和权重：`1024 * (4+4) = 8 KiB`。
- 一个输出顶点块：`256 * 4 = 1 KiB`。

主要 payload 约 74 KiB，低于 910B3 的 192 KiB UB。设置 `MAX_VERTEX_NUM=16384` 是显式容量约束，不是任意常数。

In [ ]:
%%writefile $WRITE_ROOT/op_kernel/parallel_sssp_tiling.h
#pragma once

#include <cstdint>

constexpr uint32_t MAX_VERTEX_NUM = 16384;
constexpr uint32_t VERTEX_TILE = 256;
constexpr uint32_t EDGE_TILE = 1024;
constexpr uint32_t ROW_BUFFER_ELEMENTS = ((VERTEX_TILE + 1 + 7) / 8) * 8;
constexpr float INF_DISTANCE = 1.0e30f;

struct ParallelSsspTilingData {
    uint32_t blockNum;
    uint32_t vertexCount;
    uint32_t edgeCount;
    uint32_t verticesPerCore;
};

In [ ]:
MAX_VERTEX_NUM = 16384
VERTEX_TILE = 256
EDGE_TILE = 1024
ROW_BUFFER_ELEMENTS = ((VERTEX_TILE + 1 + 7) // 8) * 8

ub_bytes = {
    "distance snapshot": MAX_VERTEX_NUM * 4,
    "row offsets": ROW_BUFFER_ELEMENTS * 4,
    "edge sources": EDGE_TILE * 4,
    "edge weights": EDGE_TILE * 4,
    "output tile": VERTEX_TILE * 4,
}
print("[步骤 4] 计算单个 AI Core 的主要 UB 占用")
for name, size in ub_bytes.items():
    print(f"{name:18s}: {size:6d} B")
print(f"total payload      : {sum(ub_bytes.values()) / 1024:.2f} KiB")
print("available UB       : 192.00 KiB")
assert sum(ub_bytes.values()) < 192 * 1024

## 5. 编写 Ascend C Pull Kernel

### Kernel 的功能

一次 Kernel Launch（核函数启动）只完成一轮同步松弛，Host需要重复启动最多 V-1 次。Init（Initialization，初始化函数）首先将输入地址绑定为 GM（Global Memory，全局内存）张量，再根据核心编号计算当前 AI Core负责的目标顶点范围，并为 UB（Unified Buffer，统一缓冲区）中的数据队列分配空间。Process（处理函数）将上一轮完整距离加载为只读快照，然后分批处理目标顶点。对于每个目标顶点 v，遍历所有前驱顶点 u，计算候选距离并选择最小值。

### 数据搬运步骤

1. 用 Ext（Extended，扩展参数） 版本 `DataCopyPad` 将 上一轮距离数组 `dist_in` 搬入 UB，处理非 32B 对齐尾部。
2. 搬入当前顶点块对应的 `row_ptr`，通过 row_ptr[v] 和 row_ptr[v+1] 得到顶点 v 的入边起止范围。。
3. 如果入边数量超过 EDGE_TILE（单次处理的最大边块大小），就将 `src_idx`（Source Index，源顶点索引）和 `weights`（边权）分批搬入 UB。
4. 从 UB 距离快照中读取前驱距离 dist[u]，计算 dist[u]+w(u,v)，再通过 Local Reduction（局部归约）从所有候选距离中选出最小值。
5. 将连续的 `dist_out` 区间用 `DataCopyPad` 写回 GM。

### 关键正确性点

- `dist_in` 只读、`dist_out` 只写，不能原地更新，否则同一轮会看到部分新值。
- 不可达距离使用 `1e30`，计算候选值前先判断前驱可达。
- GM 不采用逐元素 `GetValue/SetValue`（读取/写入单个值），因为随机访问全局内存效率较低；数据先批量搬入 UB，再在 UB 中进行索引和计算。
- `row_ptr` 和边块不足完整 Tile（分块）时，必须正确处理尾块并满足 32B 对齐要求，防止越界或数据错误。

In [ ]:
%%writefile $WRITE_ROOT/op_kernel/parallel_sssp_kernel.cpp
#include "kernel_operator.h"
#include "parallel_sssp_tiling.h"

// 每次启动 Kernel 只完成一轮同步 Pull Bellman-Ford 松弛。
class KernelParallelSssp {
public:
    __aicore__ inline KernelParallelSssp(AscendC::TPipe *pipe) : pipe_(pipe) {}

    __aicore__ inline void Init(GM_ADDR rowPtr, GM_ADDR srcIdx, GM_ADDR weights,
                                GM_ADDR distIn, GM_ADDR distOut,
                                const __gm__ ParallelSsspTilingData *tiling)
    {
        // 根据核编号为当前 AI Core 分配一段连续的目标顶点。
        tiling_ = tiling;
        const uint32_t blockIdx = AscendC::GetBlockIdx();
        startVertex_ = blockIdx * tiling_->verticesPerCore;
        const uint32_t remaining = startVertex_ < tiling_->vertexCount
            ? tiling_->vertexCount - startVertex_ : 0;
        ownedVertices_ = remaining < tiling_->verticesPerCore
            ? remaining : tiling_->verticesPerCore;

        // 将六个 GM 参数绑定为带有明确数据类型和长度的 GlobalTensor。
        rowPtrGm_.SetGlobalBuffer(reinterpret_cast<__gm__ int32_t *>(rowPtr), tiling_->vertexCount + 1);
        srcIdxGm_.SetGlobalBuffer(reinterpret_cast<__gm__ int32_t *>(srcIdx), tiling_->edgeCount);
        weightsGm_.SetGlobalBuffer(reinterpret_cast<__gm__ float *>(weights), tiling_->edgeCount);
        distInGm_.SetGlobalBuffer(reinterpret_cast<__gm__ float *>(distIn), tiling_->vertexCount);
        distOutGm_.SetGlobalBuffer(reinterpret_cast<__gm__ float *>(distOut), tiling_->vertexCount);

        // 在 UB 中为距离快照、CSR 分块和输出分块分配队列缓冲区。
        pipe_->InitBuffer(distQueue_, 1, MAX_VERTEX_NUM * sizeof(float));
        pipe_->InitBuffer(rowQueue_, 1, ROW_BUFFER_ELEMENTS * sizeof(int32_t));
        pipe_->InitBuffer(srcQueue_, 1, EDGE_TILE * sizeof(int32_t));
        pipe_->InitBuffer(weightQueue_, 1, EDGE_TILE * sizeof(float));
        pipe_->InitBuffer(outQueue_, 1, VERTEX_TILE * sizeof(float));
    }

    __aicore__ inline void Process()
    {
        // 当启动核数大于实际需要时，未分配到顶点的尾核直接返回。
        if (ownedVertices_ == 0) {
            return;
        }
        // 本轮所有松弛操作都读取同一份旧距离快照，保证同步迭代语义。
        AscendC::LocalTensor<float> distLocal = distQueue_.AllocTensor<float>();
        CopyInFloat(distLocal, distInGm_, 0, tiling_->vertexCount);
        distQueue_.EnQue(distLocal);
        distLocal = distQueue_.DeQue<float>();

        // 当前核拥有的顶点可能超过单个 UB 分块，因此按 VERTEX_TILE 分批处理。
        for (uint32_t localBase = 0; localBase < ownedVertices_; localBase += VERTEX_TILE) {
            const uint32_t remaining = ownedVertices_ - localBase;
            const uint32_t count = remaining < VERTEX_TILE ? remaining : VERTEX_TILE;
            ProcessVertexTile(distLocal, localBase, count);
        }
        distQueue_.FreeTensor(distLocal);
    }

private:
    // 扩展版 DataCopyPad 能处理长度未按 32B 对齐的尾块。
    __aicore__ inline void CopyInFloat(AscendC::LocalTensor<float> dst,
                                       AscendC::GlobalTensor<float> src,
                                       uint32_t offset, uint32_t count)
    {
        AscendC::DataCopyExtParams params{
            1, static_cast<uint32_t>(count * sizeof(float)), 0, 0, 0};
        AscendC::DataCopyPadExtParams<float> pad{false, 0, 0, 0.0f};
        AscendC::DataCopyPad(dst, src[offset], params, pad);
    }

    __aicore__ inline void CopyInInt(AscendC::LocalTensor<int32_t> dst,
                                     AscendC::GlobalTensor<int32_t> src,
                                     uint32_t offset, uint32_t count)
    {
        AscendC::DataCopyExtParams params{
            1, static_cast<uint32_t>(count * sizeof(int32_t)), 0, 0, 0};
        AscendC::DataCopyPadExtParams<int32_t> pad{false, 0, 0, 0};
        AscendC::DataCopyPad(dst, src[offset], params, pad);
    }

    __aicore__ inline void ProcessVertexTile(const AscendC::LocalTensor<float> &distLocal,
                                              uint32_t localBase, uint32_t count)
    {
        const uint32_t globalBase = startVertex_ + localBase;
        // row_ptr[v:v+count+1] 描述当前顶点块中每个目标顶点的入边区间。
        AscendC::LocalTensor<int32_t> rowLocal = rowQueue_.AllocTensor<int32_t>();
        CopyInInt(rowLocal, rowPtrGm_, globalBase, count + 1);
        rowQueue_.EnQue(rowLocal);
        rowLocal = rowQueue_.DeQue<int32_t>();

        AscendC::LocalTensor<float> outLocal = outQueue_.AllocTensor<float>();
        for (uint32_t i = 0; i < count; ++i) {
            const uint32_t vertex = globalBase + i;
            // 先保留顶点上一轮的距离，再用所有入边候选距离逐步更新最小值。
            float best = distLocal.GetValue(vertex);
            const uint32_t edgeBegin = static_cast<uint32_t>(rowLocal.GetValue(i));
            const uint32_t edgeEnd = static_cast<uint32_t>(rowLocal.GetValue(i + 1));

            // 高入度顶点的边可能无法一次放入 UB，需要按 EDGE_TILE 多次搬运。
            for (uint32_t edgeBase = edgeBegin; edgeBase < edgeEnd; edgeBase += EDGE_TILE) {
                const uint32_t edgeRemaining = edgeEnd - edgeBase;
                const uint32_t edgeCount = edgeRemaining < EDGE_TILE ? edgeRemaining : EDGE_TILE;
                AscendC::LocalTensor<int32_t> srcLocal = srcQueue_.AllocTensor<int32_t>();
                AscendC::LocalTensor<float> weightLocal = weightQueue_.AllocTensor<float>();
                CopyInInt(srcLocal, srcIdxGm_, edgeBase, edgeCount);
                CopyInFloat(weightLocal, weightsGm_, edgeBase, edgeCount);
                srcQueue_.EnQue(srcLocal);
                weightQueue_.EnQue(weightLocal);
                srcLocal = srcQueue_.DeQue<int32_t>();
                weightLocal = weightQueue_.DeQue<float>();

                // Pull 松弛公式：dist_out[v] = min(dist_in[v], dist_in[u] + w(u, v))。
                for (uint32_t j = 0; j < edgeCount; ++j) {
                    const int32_t src = srcLocal.GetValue(j);
                    const float srcDistance = distLocal.GetValue(static_cast<uint32_t>(src));
                    if (srcDistance < INF_DISTANCE * 0.5f) {
                        const float candidate = srcDistance + weightLocal.GetValue(j);
                        best = candidate < best ? candidate : best;
                    }
                }
                srcQueue_.FreeTensor(srcLocal);
                weightQueue_.FreeTensor(weightLocal);
            }
            outLocal.SetValue(i, best);
        }

        outQueue_.EnQue(outLocal);
        outLocal = outQueue_.DeQue<float>();
        AscendC::DataCopyExtParams outParams{
            1, static_cast<uint32_t>(count * sizeof(float)), 0, 0, 0};
        // 每个目标顶点只由一个核负责，因此连续写回不会产生跨核写冲突。
        AscendC::DataCopyPad(distOutGm_[globalBase], outLocal, outParams);
        outQueue_.FreeTensor(outLocal);
        rowQueue_.FreeTensor(rowLocal);
    }

    AscendC::TPipe *pipe_;
    const __gm__ ParallelSsspTilingData *tiling_;
    AscendC::GlobalTensor<int32_t> rowPtrGm_, srcIdxGm_;
    AscendC::GlobalTensor<float> weightsGm_, distInGm_, distOutGm_;
    AscendC::TQue<AscendC::TPosition::VECIN, 1> distQueue_, rowQueue_, srcQueue_, weightQueue_;
    AscendC::TQue<AscendC::TPosition::VECOUT, 1> outQueue_;
    uint32_t startVertex_ = 0;
    uint32_t ownedVertices_ = 0;
};

// ascendc_library 根据该入口函数生成供 Host 调用的 Kernel 启动头文件。
extern "C" __global__ __aicore__ void parallel_sssp_kernel(
    GM_ADDR rowPtr, GM_ADDR srcIdx, GM_ADDR weights,
    GM_ADDR distIn, GM_ADDR distOut, GM_ADDR tiling)
{
    AscendC::TPipe pipe;
    KernelParallelSssp op(&pipe);
    op.Init(rowPtr, srcIdx, weights, distIn, distOut,
            reinterpret_cast<__gm__ ParallelSsspTilingData *>(tiling));
    op.Process();
}

## 6. 编写 Host 侧二进制工具

`data_utils.h` 封装二进制数组的读取、写出和文件大小检查。它把与算法无关的文件 I/O 从主程序中分离，使 `parallel_sssp_main.cpp` 更聚焦于 算子调度。

二进制文件长度必须是元素类型大小的整数倍；空文件、打开失败和读取不完整都要立即报错，不能把不完整数据继续传给设备。

In [ ]:
%%writefile $WRITE_ROOT/op_host/data_utils.h
#pragma once

#include <fstream>
#include <stdexcept>
#include <string>
#include <vector>

template <typename T>
std::vector<T> ReadBinary(const std::string &path, size_t count)
{
    std::ifstream file(path, std::ios::binary | std::ios::ate);
    if (!file) {
        throw std::runtime_error("cannot open " + path);
    }
    const auto bytes = static_cast<size_t>(file.tellg());
    if (bytes != count * sizeof(T)) {
        throw std::runtime_error("unexpected byte size for " + path);
    }
    file.seekg(0);
    std::vector<T> data(count);
    file.read(reinterpret_cast<char *>(data.data()), static_cast<std::streamsize>(bytes));
    return data;
}

template <typename T>
void WriteBinary(const std::string &path, const std::vector<T> &data)
{
    std::ofstream file(path, std::ios::binary | std::ios::trunc);
    if (!file) {
        throw std::runtime_error("cannot create " + path);
    }
    file.write(reinterpret_cast<const char *>(data.data()),
               static_cast<std::streamsize>(data.size() * sizeof(T)));
}

## 7. 编写 Host 调度程序

### Host 负责什么

Host 是完整应用的控制：读取文件、验证 CSR、初始化 ACL 与设备、申请 GM、拷贝输入、计算 Tiling、通过 `ACLRT_LAUNCH_KERNEL` 循环启动独立编译的 Kernel、同步轮次、拷回结果并释放资源。

### 为什么需要 `V-1` 轮与 stream 同步

任意不含重复顶点的最短路最多包含 `V-1` 条边，因此 Bellman-Ford 最坏需要 `V-1` 轮。每轮 Kernel 读取同一个旧快照；`aclrtSynchronizeStream` 保证上一轮全部核心写完后，Host 才交换缓冲区并启动下一轮。单个 Kernel 内没有跨核全局屏障，所以不能把所有轮次塞进一次 launch。

Host 会再次检查 `row_ptr[0]=0`、单调性、末项等于 `E`、源顶点范围、权重非负有限、顶点数上限与源点范围。Python 生成器和 Host 双重校验，是为了让程序即使接收外部二进制数据也不会越界访问。

In [ ]:
%%writefile $WRITE_ROOT/op_host/parallel_sssp_main.cpp
#include <algorithm>
#include <cmath>
#include <cstdint>
#include <iostream>
#include <stdexcept>
#include <string>
#include <utility>
#include <vector>

#include "acl/acl.h"
#include "aclrtlaunch_parallel_sssp_kernel.h"
#include "data_utils.h"
#include "../op_kernel/parallel_sssp_tiling.h"

#define ACL_CHECK(call) do { \
    const aclError ret = (call); \
    if (ret != ACL_SUCCESS) { \
        throw std::runtime_error(std::string(#call) + " failed: " + std::to_string(ret)); \
    } \
} while (0)

template <typename T>
uint8_t *MallocAndCopy(const std::vector<T> &host)
{
    // 设备地址使用 uint8_t*，以匹配自动生成的 ACLRT Kernel 启动接口。
    uint8_t *device = nullptr;
    const size_t dataBytes = host.size() * sizeof(T);
    const size_t allocationBytes = std::max(sizeof(T), dataBytes);
    ACL_CHECK(aclrtMalloc(reinterpret_cast<void **>(&device), allocationBytes,
                         ACL_MEM_MALLOC_HUGE_FIRST));
    if (dataBytes != 0) {
        ACL_CHECK(aclrtMemcpy(device, allocationBytes, host.data(), dataBytes,
                             ACL_MEMCPY_HOST_TO_DEVICE));
    }
    return device;
}

void ValidateGraph(const std::vector<int32_t> &rowPtr,
                   const std::vector<int32_t> &srcIdx,
                   const std::vector<float> &weights,
                   uint32_t vertexCount, uint32_t edgeCount, uint32_t source)
{
    // 在访问 NPU 内存前拒绝结构错误的 CSR，避免非法数据进入设备侧计算。
    if (vertexCount == 0 || vertexCount > MAX_VERTEX_NUM) {
        throw std::invalid_argument("vertexCount must be in [1, MAX_VERTEX_NUM]");
    }
    if (source >= vertexCount || rowPtr.size() != vertexCount + 1 ||
        srcIdx.size() != edgeCount || weights.size() != edgeCount) {
        throw std::invalid_argument("shape or source validation failed");
    }
    if (rowPtr.front() != 0 || rowPtr.back() != static_cast<int32_t>(edgeCount)) {
        throw std::invalid_argument("row_ptr endpoints are invalid");
    }
    for (uint32_t i = 0; i < vertexCount; ++i) {
        if (rowPtr[i] > rowPtr[i + 1]) {
            throw std::invalid_argument("row_ptr must be nondecreasing");
        }
    }
    for (uint32_t i = 0; i < edgeCount; ++i) {
        if (srcIdx[i] < 0 || srcIdx[i] >= static_cast<int32_t>(vertexCount)) {
            throw std::invalid_argument("src_idx contains an invalid vertex");
        }
        if (!std::isfinite(weights[i]) || weights[i] < 0.0f) {
            throw std::invalid_argument("weights must be finite and nonnegative");
        }
    }
}

int main(int argc, char **argv)
{
    // 从命令行读取顶点数、边数和源点编号，三者由 meta.txt 提供。
    if (argc != 4) {
        std::cerr << "usage: parallel_sssp <vertex_count> <edge_count> <source>\n";
        return 2;
    }
    const uint32_t vertexCount = static_cast<uint32_t>(std::stoul(argv[1]));
    const uint32_t edgeCount = static_cast<uint32_t>(std::stoul(argv[2]));
    const uint32_t source = static_cast<uint32_t>(std::stoul(argv[3]));

    try {
        // 读取转置 CSR 输入，并在分配设备内存前完成形状、索引和边权校验。
        std::cout << "[Host 1/5] Reading input/{row_ptr,src_idx,weights}.bin\n";
        const auto rowPtr = ReadBinary<int32_t>("input/row_ptr.bin", vertexCount + 1);
        const auto srcIdx = ReadBinary<int32_t>("input/src_idx.bin", edgeCount);
        const auto weights = ReadBinary<float>("input/weights.bin", edgeCount);
        ValidateGraph(rowPtr, srcIdx, weights, vertexCount, edgeCount, source);
        std::cout << "           Graph validated: V=" << vertexCount
                  << ", E=" << edgeCount << ", source=" << source << "\n";

        // 初始化 ACL、选择 0 号设备并创建用于顺序提交 Kernel 的执行流。
        std::cout << "[Host 2/5] Initializing ACL device 0 and stream\n";
        ACL_CHECK(aclInit(nullptr));
        ACL_CHECK(aclrtSetDevice(0));
        aclrtStream stream = nullptr;
        ACL_CHECK(aclrtCreateStream(&stream));

        // 查询可用 Vector Core 数量，按目标顶点数量确定实际启动核数。
        int64_t availableCores = 0;
        ACL_CHECK(aclrtGetDeviceInfo(0, ACL_DEV_ATTR_VECTOR_CORE_NUM, &availableCores));
        if (availableCores <= 0) {
            throw std::runtime_error("device reports no available Vector Core");
        }
        const uint32_t blockNum = std::min<uint32_t>(
            vertexCount, static_cast<uint32_t>(availableCores));
        // Tiling 数据描述图规模、启动核数及每个核最多处理的顶点数量。
        ParallelSsspTilingData tiling{
            blockNum, vertexCount, edgeCount,
            (vertexCount + blockNum - 1) / blockNum
        };
        std::cout << "[Host 3/5] Tiling: blocks=" << blockNum
                  << ", vertices/core=" << tiling.verticesPerCore << "\n";

        std::cout << "[Host 4/5] Copying CSR and distance buffers to NPU; launching "
                  << (vertexCount - 1) << " synchronous rounds\n";
        // 将只读 CSR、初始距离和 Tiling 数据搬入 GM，并准备双距离缓冲区。
        uint8_t *rowPtrDevice = MallocAndCopy(rowPtr);
        uint8_t *srcIdxDevice = MallocAndCopy(srcIdx);
        uint8_t *weightsDevice = MallocAndCopy(weights);
        std::vector<float> initial(vertexCount, INF_DISTANCE);
        initial[source] = 0.0f;
        uint8_t *distInDevice = MallocAndCopy(initial);
        uint8_t *distOutDevice = nullptr;
        ACL_CHECK(aclrtMalloc(reinterpret_cast<void **>(&distOutDevice),
                             vertexCount * sizeof(float), ACL_MEM_MALLOC_HUGE_FIRST));
        uint8_t *tilingDevice = nullptr;
        ACL_CHECK(aclrtMalloc(reinterpret_cast<void **>(&tilingDevice),
                             sizeof(tiling), ACL_MEM_MALLOC_HUGE_FIRST));
        ACL_CHECK(aclrtMemcpy(tilingDevice, sizeof(tiling), &tiling, sizeof(tiling),
                             ACL_MEMCPY_HOST_TO_DEVICE));

        // 最多执行 V-1 轮；每轮同步后交换输入输出指针，保持严格的双缓冲语义。
        for (uint32_t iteration = 0; iteration + 1 < vertexCount; ++iteration) {
            ACLRT_LAUNCH_KERNEL(parallel_sssp_kernel)(blockNum, stream,
                rowPtrDevice, srcIdxDevice, weightsDevice,
                distInDevice, distOutDevice, tilingDevice);
            ACL_CHECK(aclrtSynchronizeStream(stream));
            std::swap(distInDevice, distOutDevice);
        }

        // 循环结束后 distInDevice 指向最终有效结果，将其复制回 Host 并写入文件。
        std::vector<float> output(vertexCount);
        ACL_CHECK(aclrtMemcpy(output.data(), output.size() * sizeof(float), distInDevice,
                             output.size() * sizeof(float), ACL_MEMCPY_DEVICE_TO_HOST));
        WriteBinary("output/output.bin", output);
        std::cout << "[Host 5/5] Copied result to output/output.bin ("
                  << output.size() << " float32 values)\n";

        // 按与创建相反的顺序释放设备内存、执行流和 ACL 运行环境。
        aclrtFree(tilingDevice);
        aclrtFree(distOutDevice);
        aclrtFree(distInDevice);
        aclrtFree(weightsDevice);
        aclrtFree(srcIdxDevice);
        aclrtFree(rowPtrDevice);
        aclrtDestroyStream(stream);
        aclrtResetDevice(0);
        aclFinalize();
        std::cout << "SSSP execution completed successfully.\n";
        return 0;
    } catch (const std::exception &e) {
        std::cerr << "ERROR: " << e.what() << "\n";
        return 1;
    }
}

## 8. 编写精度验证与异常输入测试

### 正常结果如何验证

不可达顶点用大数表示，直接纳入相对误差会掩盖错误，因此先比较“是否可达”的布尔掩码。仅对有限距离计算：

`MERE = mean(|actual-golden| / (|golden| + 1e-7))`

`MARE = max(|actual-golden| / (|golden| + 1e-7))`

验收阈值为 `MERE < 2^-13` 且 `MARE < 10 * 2^-13`。

### 异常测试为什么独立

负权、NaN、Inf、越界顶点和非法 `row_ptr` 都必须在进入 Kernel 前拒绝。

In [ ]:
%%writefile $WRITE_ROOT/scripts/verify_result.py
from pathlib import Path

import numpy as np

INF = np.float32(1.0e30)
THRESHOLD = 2.0 ** -13


def main():
    # meta.txt 决定本案例需要读取的有效顶点数量。
    vertex_count, _, _ = map(int, Path("meta.txt").read_text().split())
    output = np.fromfile("output/output.bin", dtype=np.float32, count=vertex_count)
    golden = np.fromfile("output/golden.bin", dtype=np.float32, count=vertex_count)
    if output.size != vertex_count or golden.size != vertex_count:
        raise SystemExit("FAILED: output or golden size mismatch")
    # 先比较可达性，避免用普通浮点误差指标比较不可达标记 1e30。
    output_unreachable = output >= INF * 0.5
    golden_unreachable = golden >= INF * 0.5
    reachability_ok = np.array_equal(output_unreachable, golden_unreachable)
    finite = ~golden_unreachable
    # 仅对可达顶点计算 MERE（平均相对误差）与 MARE（最大相对误差）。
    if np.any(finite):
        relative = np.abs(output[finite] - golden[finite]) / (
            np.abs(golden[finite]) + 1.0e-7)
        mere, mare = float(np.mean(relative)), float(np.max(relative))
    else:
        mere = mare = 0.0
    passed = reachability_ok and mere < THRESHOLD and mare < 10 * THRESHOLD
    print(f"reachability_ok={reachability_ok}, MERE={mere:.8e}, MARE={mare:.8e}")
    print(f"thresholds: MERE<{THRESHOLD:.8e}, MARE<{10 * THRESHOLD:.8e}")
    if not passed:
        print("output:", output)
        print("golden:", golden)
        raise SystemExit("FAILED")
    print("PASSED")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile $WRITE_ROOT/scripts/validate_inputs.py
import math

from graph_utils import validate_edges


invalid_cases = [
    (3, [(0, 3, 1.0)]),
    (3, [(0, 1, -1.0)]),
    (3, [(0, 1, math.inf)]),
    (3, [(0, 1, math.nan)]),
]
for vertex_count, edges in invalid_cases:
    try:
        validate_edges(vertex_count, edges)
    except ValueError as exc:
        print("rejected as expected:", exc)
    else:
        raise AssertionError(f"invalid case was accepted: {edges}")
print("input validation tests PASSED")

In [ ]:
validation_script = WORK_DIR / "scripts" / "validate_inputs.py"
print("[步骤 8] 执行非法输入测试")
print("脚本：", validation_script)
subprocess.run(
    [sys.executable, str(validation_script)],
    cwd=WORK_DIR,
    check=True,
)
print("状态：所有非法输入均按预期被拒绝。")

## 9. 编写 CMake 构建配置

`CMakeLists.txt` 将两侧分开构建。

CMake 必须在加载 CANN 环境后执行；如 Toolkit 不在标准路径，应以当前 CANNLab 模板提供的环境变量为准。

In [ ]:
%%writefile $WRITE_ROOT/CMakeLists.txt
cmake_minimum_required(VERSION 3.16)
project(parallel_sssp LANGUAGES CXX)

set(CMAKE_CXX_STANDARD 17)
set(CMAKE_CXX_STANDARD_REQUIRED ON)
set(CMAKE_CXX_EXTENSIONS OFF)

if(NOT CMAKE_BUILD_TYPE)
  set(CMAKE_BUILD_TYPE Release CACHE STRING "Build type" FORCE)
endif()

set(SOC_VERSION "ascend910b3" CACHE STRING "Ascend SoC version")
set(RUN_MODE "npu" CACHE STRING "Ascend C run mode")
set(ASCEND_CANN_PATH "$ENV{ASCEND_HOME_PATH}" CACHE PATH "CANN installation path")
if(NOT ASCEND_CANN_PATH)
  set(ASCEND_CANN_PATH "$ENV{ASCEND_TOOLKIT_HOME}" CACHE PATH "CANN installation path" FORCE)
endif()
if(NOT ASCEND_CANN_PATH)
  message(FATAL_ERROR "ASCEND_HOME_PATH or ASCEND_TOOLKIT_HOME is not set")
endif()

set(ASCEND_CANN_PACKAGE_PATH "${ASCEND_CANN_PATH}" CACHE PATH "CANN package path" FORCE)
set(CMAKE_INSTALL_PREFIX "${CMAKE_BINARY_DIR}/out" CACHE PATH "Ascend C output path" FORCE)

set(ASCENDC_CMAKE_CANDIDATES
  "${ASCEND_CANN_PACKAGE_PATH}/tools/tikcpp/ascendc_kernel_cmake/ascendc.cmake"
  "${ASCEND_CANN_PACKAGE_PATH}/compiler/tikcpp/ascendc_kernel_cmake/ascendc.cmake"
  "${ASCEND_CANN_PACKAGE_PATH}/aarch64-linux/tikcpp/ascendc_kernel_cmake/ascendc.cmake"
  "${ASCEND_CANN_PACKAGE_PATH}/x86_64-linux/tikcpp/ascendc_kernel_cmake/ascendc.cmake"
)
foreach(candidate IN LISTS ASCENDC_CMAKE_CANDIDATES)
  if(EXISTS "${candidate}")
    set(ASCENDC_CMAKE_FILE "${candidate}")
    break()
  endif()
endforeach()
if(NOT ASCENDC_CMAKE_FILE)
  message(FATAL_ERROR "Cannot find ascendc.cmake under ${ASCEND_CANN_PACKAGE_PATH}")
endif()

message(STATUS "ASCEND_CANN_PACKAGE_PATH=${ASCEND_CANN_PACKAGE_PATH}")
message(STATUS "SOC_VERSION=${SOC_VERSION}")
include("${ASCENDC_CMAKE_FILE}")

# 单独编译 Ascend C 入口，并自动生成对应的 ACLRT Kernel 启动头文件。
ascendc_library(parallel_sssp_kernels STATIC
  op_kernel/parallel_sssp_kernel.cpp
)
ascendc_include_directories(parallel_sssp_kernels PRIVATE
  ${CMAKE_CURRENT_SOURCE_DIR}/op_kernel
)
ascendc_compile_definitions(parallel_sssp_kernels PRIVATE
  -DASCENDC_DUMP=0
)

# 将控制程序按普通 C++ 编译，再与 Kernel 目标及 CANN 运行库链接。
add_executable(parallel_sssp
  op_host/parallel_sssp_main.cpp
)
target_include_directories(parallel_sssp PRIVATE
  op_host
  op_kernel
  ${ASCEND_CANN_PACKAGE_PATH}/include
  ${ASCEND_CANN_PACKAGE_PATH}/include/external
  ${ASCEND_CANN_PACKAGE_PATH}/runtime/include
  ${CMAKE_INSTALL_PREFIX}/include/parallel_sssp_kernels
  ${CMAKE_BINARY_DIR}/out/include/parallel_sssp_kernels
)
target_link_directories(parallel_sssp PRIVATE
  ${ASCEND_CANN_PACKAGE_PATH}/lib64
  ${ASCEND_CANN_PACKAGE_PATH}/runtime/lib64/stub
  ${ASCEND_CANN_PACKAGE_PATH}/runtime/lib64
  ${ASCEND_CANN_PACKAGE_PATH}/acllib/lib64
  ${ASCEND_CANN_PACKAGE_PATH}/aarch64-linux/devlib
  ${ASCEND_CANN_PACKAGE_PATH}/x86_64-linux/devlib
)
target_compile_definitions(parallel_sssp PRIVATE
  SOC_VERSION="${SOC_VERSION}"
)
target_link_libraries(parallel_sssp PRIVATE
  parallel_sssp_kernels
  ascendcl
  tiling_api
  register
  platform
  ascendalog
  c_sec
  dl
)
add_dependencies(parallel_sssp parallel_sssp_kernels)

## 10. 编写一键运行脚本与工程说明

`run.sh` 把标准验收流程固化为：加载环境 → CMake 配置 → 并行编译 → 逐案例生成数据 → 执行 NPU 程序 → 对比 Golden。

脚本使用 `set -euo pipefail`，任一命令失败便立即停止，防止后续步骤基于无效产物继续运行。

In [ ]:
%%writefile $WRITE_ROOT/run.sh
#!/usr/bin/env bash
set -euo pipefail

SCRIPT_DIR="$(cd "$(dirname "${BASH_SOURCE[0]}")" && pwd)"
cd "${SCRIPT_DIR}"
: "${ASCEND_HOME_PATH:?ASCEND_HOME_PATH is not set}"
source "${ASCEND_HOME_PATH}/set_env.sh"

# 从干净的构建目录开始，避免旧 CMake 缓存影响本次运行。
rm -rf build
mkdir -p build
cd build
cmake .. -DSOC_VERSION="${SOC_VERSION:-ascend910b3}"
make -j4

cases=(demo single chain disconnected zero_weight tie random_sparse random_dense)
for case_name in "${cases[@]}"; do
    echo "=== case: ${case_name} ==="
    # 生成转置 CSR 输入，并使用独立的 CPU Dijkstra 计算 Golden 参考结果。
    python3 ../scripts/gen_data.py --case "${case_name}"
    read -r vertex_count edge_count source < meta.txt
    rm -f output/output.bin
    # 运行 C++ Host，由 Host 最多启动 V-1 轮 NPU Kernel。
    ./parallel_sssp "${vertex_count}" "${edge_count}" "${source}"
    test -f output/output.bin
    # 将 NPU 输出与 Golden 比较，通过后再进入下一个图案例。
    python3 ../scripts/verify_result.py
done
python3 ../scripts/validate_inputs.py
echo "All parallel SSSP cases PASSED"

In [ ]:
if os.name != "nt":
    subprocess.run(["chmod", "+x", "run.sh"], cwd=WORK_DIR, check=True)

print("[步骤 10] 工程文件生成完成")
print("下面展示 Notebook 已创建的目录和文件：")
for root, dirs, files in os.walk(WORK_DIR):
    dirs[:] = sorted(d for d in dirs if d != "CMakeFiles")
    level = len(Path(root).relative_to(WORK_DIR).parts)
    if level <= 2:
        print("  " * level + Path(root).name + "/")
        for filename in sorted(files):
            print("  " * (level + 1) + filename)

## 11. 在 Notebook 中执行 CMake 编译

下面的 code cell 就是正式构建步骤，不需要离开 Notebook 手工编辑或打开终端。它展示参与构建的 Kernel、Host 和 CMake 文件，在空的 `build` 目录中完成配置与编译，并列出生成的启动头文件和可执行程序。完整构建输出同时保存为 `build.log`。

- CMake 配置阶段应找到 `ascendc.cmake` 并识别 `ascend910b3`。
- Kernel 目标会生成 `aclrtlaunch_parallel_sssp_kernel.h`，Host 通过该头文件启动 Kernel。
- 编译阶段先生成 Kernel，再链接 Host 可执行文件 `parallel_sssp`。
- 若提示找不到 Ascend C CMake 模块，通常是 CANN 环境没有加载，而不是算法源码错误。
- 修改任一 `%%writefile` 单元后，重新执行本单元即可增量构建。

In [ ]:
NPU_READY = bool(os.environ.get("ASCEND_HOME_PATH") and shutil.which("npu-smi"))
if NPU_READY:
    kernel_path = WORK_DIR / "op_kernel" / "parallel_sssp_kernel.cpp"
    host_path = WORK_DIR / "op_host" / "parallel_sssp_main.cpp"
    build_dir = WORK_DIR / "build"

    print("[构建 1/4] 确认参与构建的文件")
    print("Kernel:", kernel_path)
    print("Host  :", host_path)
    print("CMake :", WORK_DIR / "CMakeLists.txt")

    print("\n[构建 2/4] 创建干净的 build 目录")
    # 清理旧缓存，避免之前生成的启动头文件或目标文件影响本次结果。
    shutil.rmtree(build_dir, ignore_errors=True)
    build_dir.mkdir(parents=True, exist_ok=True)
    # CANN 环境加载、CMake 配置和编译必须在同一个 Bash 子进程中完成。
    build_command = (
        'source "$ASCEND_HOME_PATH/set_env.sh" && '
        'cmake .. -DSOC_VERSION=ascend910b3 && '
        'cmake --build . --clean-first -j4'
    )
    print("命令：cmake .. -DSOC_VERSION=ascend910b3")
    print("命令：cmake --build . --clean-first -j4")

    print("\n[构建 3/4] 配置并编译 Kernel 与 C++ Host")
    result = subprocess.run(
        ["bash", "-lc", build_command],
        cwd=build_dir,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )
    # 完整输出保存到日志文件；Notebook 仅展示成功尾部或失败关键信息。
    log_path = build_dir / "build.log"
    log_path.write_text(result.stdout, encoding="utf-8")
    build_lines = result.stdout.splitlines()
    if result.returncode != 0:
        print("\n".join(build_lines[-60:]))
        raise RuntimeError(f"编译失败，完整日志：{log_path}")
    print("\n".join(build_lines[-25:]))

    print("\n[构建 4/4] 检查构建产物")
    launch_headers = sorted(build_dir.rglob("aclrtlaunch_parallel_sssp_kernel.h"))
    executable = build_dir / "parallel_sssp"
    print("启动头文件：", launch_headers[0] if launch_headers else "由构建目标内部生成")
    print("可执行程序：", executable)
    print("完整日志  ：", log_path)
    print("状态：构建成功，可以进入单案例运行。")
else:
    print("跳过 NPU 编译：请在 CANNLab 910B3 环境加载 CANN 后重跑本单元。")

## 12. 在 Notebook 中执行单个案例

这一单元把运行链路显式拆开：选择案例 → 生成数据 → 读取 `meta.txt` → 启动 Host 程序 → 运行精度验证。修改 code cell 第一行的 `CASE_NAME`，即可交互式选择 `demo`、`single`、`chain`、`disconnected`、`zero_weight`、`tie`、`random_sparse` 或 `random_dense`。

Host 命令行参数依次给出 `V`、`E` 和源点；文件位置采用工程约定的 `input/*.bin` 与 `output/output.bin`。程序内部最多进行 `V-1` 次同步 Kernel launch，并把最终有效缓冲区写入结果文件。

In [ ]:
CASE_NAME = "demo"  # 交互入口：可改为 chain、zero_weight、random_sparse 等

if NPU_READY:
    build_dir = WORK_DIR / "build"
    generator = WORK_DIR / "scripts" / "gen_data.py"
    verifier = WORK_DIR / "scripts" / "verify_result.py"

    # 每次切换案例都重新生成输入，确保二进制数据、Golden 与 meta.txt 相互匹配。
    print(f"[运行 1/4] 生成 {CASE_NAME} 图、转置 CSR 和 CPU Golden")
    print("调用：", generator, "--case", CASE_NAME)
    subprocess.run(
        [sys.executable, str(generator), "--case", CASE_NAME],
        cwd=build_dir, check=True,
    )
    # Host 的三个命令行参数直接取自本案例的元数据。
    vertex_count, edge_count, source = map(
        int, (build_dir / "meta.txt").read_text(encoding="utf-8").split()
    )
    print(f"图信息：V={vertex_count}, E={edge_count}, source={source}")

    print("\n[运行 2/4] 调用 C++ Host；Host 将启动 NPU Kernel")
    run_command = (
        'source "$ASCEND_HOME_PATH/set_env.sh" && '
        f'./parallel_sssp {vertex_count} {edge_count} {source}'
    )
    print(f"命令：./parallel_sssp {vertex_count} {edge_count} {source}")
    subprocess.run(["bash", "-lc", run_command], cwd=build_dir, check=True)

    output_path = build_dir / "output" / "output.bin"
    print("\n[运行 3/4] 读取 NPU 输出")
    print("输出文件：", output_path)
    npu_output = np.fromfile(output_path, dtype=np.float32)
    print("NPU distances:", npu_output)

    # 验证脚本分别检查可达掩码与有限距离误差，二者均通过才算成功。
    print("\n[运行 4/4] 与 CPU Golden 比较")
    print("调用：", verifier)
    subprocess.run([
        sys.executable, str(verifier),
    ], cwd=build_dir, check=True)
    print(f"状态：{CASE_NAME} 案例运行和精度验证完成。")
else:
    print("跳过 NPU 单案例执行；CPU 数据生成与 Golden 已在第 3 节验证。")

## 13. 执行完整回归测试

最后运行 `run.sh`，覆盖 `demo`、单顶点、链式图、不可达顶点、零权边、等长最短路、稀疏随机图和较密随机图。每个案例都重新生成输入并独立比较 Golden。


In [ ]:
if NPU_READY:
    print("[完整回归] 调用一键脚本：", WORK_DIR / "run.sh")
    print("测试案例：demo、single、chain、disconnected、zero_weight、tie、random_sparse、random_dense")
    print("每个案例依次执行：生成输入 -> NPU SSSP -> Golden 比较\n")
    subprocess.run(["bash", "run.sh"], cwd=WORK_DIR, check=True)
    print("\n状态：8 个图案例和非法输入测试全部完成。")
else:
    print("跳过完整 NPU 回归；请在目标 CANNLab 环境执行本单元。")


完成后继续进入 [05.03 章节实践](05.03_chapter_practice.ipynb)。

## 课后思考

1. 为什么 Pull 写回不需要 `AtomicMin`，而 Push 通常存在写冲突？
2. 若图的最大顶点数增大到 65536，完整距离快照还能否放入 UB？你会如何改造？
3. 为什么 Host 必须在相邻两轮 Kernel 之间同步 stream？
4. 按顶点均分遇到幂律图时可能出现什么负载问题？

In [ ]:
from pathlib import Path

p = Path("answer/05.02_parallel_sssp/answers.md")
if not p.exists():
    p = Path("05_图邻接表向CSR稀疏张量的格式转换") / p

print(p.read_text(encoding="utf-8"))